# Concurrency

**Objective:** Scale capacity by running independently scalable web and worker process types.

## Simple version

Web processes serve HTTP while worker processes consume queued jobs.

In [ ]:
process_types = {
    "web": {"replicas": 2, "concurrency_each": 1},
    "worker": {"replicas": 3, "concurrency_each": 10},
}

print(process_types)

## Polished version

A scale plan makes capacity explicit and changes replica counts without changing application code.

In [ ]:
from dataclasses import dataclass, replace


@dataclass(frozen=True)
class ProcessType:
    name: str
    command: tuple[str, ...]
    replicas: int
    concurrency_each: int

    @property
    def capacity(self) -> int:
        return self.replicas * self.concurrency_each


@dataclass(frozen=True)
class ScalePlan:
    processes: tuple[ProcessType, ...]

    def scale(self, name: str, replicas: int) -> "ScalePlan":
        if replicas < 0:
            raise ValueError("replicas cannot be negative")
        updated = tuple(
            replace(process, replicas=replicas)
            if process.name == name
            else process
            for process in self.processes
        )
        if updated == self.processes:
            raise KeyError(name)
        return ScalePlan(updated)


plan = ScalePlan(
    processes=(
        ProcessType("web", ("uvicorn", "app.main:app"), 2, 1),
        ProcessType("worker", ("arq", "app.worker.WorkerSettings"), 3, 10),
    )
)
scaled = plan.scale("worker", replicas=5)

for process in scaled.processes:
    print(process.name, "capacity:", process.capacity)

## Applied in this repository

The LLM API defines separate web and worker commands. ARQ limits concurrency inside each worker, while deployment replica counts scale total capacity.